In [1]:
# Notebook bootstrap: download required files from GitHub if missing
from pathlib import Path
from urllib.request import urlretrieve

GITHUB_RAW_BASE = "https://raw.githubusercontent.com/MehrdadJalali-AI/data-management-teaching-materials/main"

required_files = [
    ("data/metadata/climate_dataset_schemaorg.jsonld", "data/metadata/climate_dataset_schemaorg.jsonld"),
]

for local_rel, github_rel in required_files:
    local_path = Path(local_rel)
    if not local_path.exists():
        local_path.parent.mkdir(parents=True, exist_ok=True)
        url = f"{GITHUB_RAW_BASE}/{github_rel}"
        urlretrieve(url, local_path)
        print(f"Downloaded: {local_path}")

print("Bootstrap complete.")

Downloaded: data/metadata/climate_dataset_schemaorg.jsonld
Bootstrap complete.


# Day 2: Process schema.org JSON-LD (Offline)

In many portals, metadata is represented as **JSON-LD** using schema.org vocabulary. In this notebook we:
- load a local schema.org JSON-LD file,
- inspect the record structure,
- flatten selected fields into a small table,
- and highlight discovery-related fields for Findability.

### Colab tip
If `../../data/metadata/...` is not found, upload `climate_dataset_schemaorg.jsonld` and the notebook will use `/content/climate_dataset_schemaorg.jsonld`.


In [3]:
from pathlib import Path
import json
import pandas as pd

# ---- Notebook-local schema.org parser (no src dependency) ----
def parse_schemaorg_jsonld(jsonld_path):
    obj = json.loads(Path(jsonld_path).read_text(encoding='utf-8'))
    ds = obj
    if isinstance(obj, dict) and '@graph' in obj and isinstance(obj['@graph'], list):
        ds = next((x for x in obj['@graph'] if isinstance(x, dict) and x.get('@type') == 'Dataset'), obj)

    creator_value = ds.get('creator', [])
    creators = []
    if isinstance(creator_value, list):
        for c in creator_value:
            if isinstance(c, dict):
                n = c.get('name')
                if n:
                    creators.append(n)
            elif c:
                creators.append(str(c))
    elif isinstance(creator_value, dict):
        n = creator_value.get('name')
        if n:
            creators.append(n)

    publisher_value = ds.get('publisher')
    publisher = publisher_value.get('name') if isinstance(publisher_value, dict) else publisher_value

    keywords = ds.get('keywords', [])
    if isinstance(keywords, str):
        keywords = [keywords]

    format_value = None
    distribution = ds.get('distribution')
    if isinstance(distribution, list) and distribution:
        first = distribution[0]
        if isinstance(first, dict):
            format_value = first.get('encodingFormat')

    return {
        'standard': 'schema.org',
        'type': ds.get('@type'),
        'title': ds.get('name'),
        'creator': creators,
        'identifier': ds.get('identifier'),
        'publisher': publisher,
        'description': ds.get('description'),
        'keywords': keywords,
        'license': ds.get('license'),
        'format': format_value,
    }

# Path strategy for local repo or Colab upload
default_path = Path('/content/data/metadata/climate_dataset_schemaorg.jsonld')
schema_path = default_path if default_path.exists() else Path('/content/data/metadata/climate_dataset_schemaorg.jsonld')

raw_obj = json.loads(schema_path.read_text(encoding="utf-8"))
list(raw_obj.keys())

['@context',
 '@type',
 'name',
 'description',
 'creator',
 'publisher',
 'identifier',
 'keywords',
 'license',
 'distribution']

In [4]:
meta = parse_schemaorg_jsonld(schema_path)
meta

{'standard': 'schema.org',
 'type': 'Dataset',
 'title': '2025 Global Climate Data',
 'creator': ['Global Climate Data Team'],
 'identifier': '10.1234/gcd-2025',
 'publisher': 'Open Research Repository (Teaching)',
 'description': 'A small teaching dataset containing fictional annual climate summaries. Variables include average temperature, rainfall, and CO2 emissions for two fictional countries (Alandia and Borealia) from 2021 to 2023.',
 'keywords': ['climate',
  'temperature',
  'rainfall',
  'CO2 emissions',
  'teaching dataset'],
 'license': 'https://creativecommons.org/licenses/by/4.0/',
 'format': 'text/csv'}

In [5]:
flattened = {
    "@type": meta.get("type"),
    "title": meta.get("title"),
    "creator": "; ".join(meta.get("creator") or []),
    "identifier": meta.get("identifier"),
    "publisher": meta.get("publisher"),
    "description": meta.get("description"),
    "keywords": "; ".join(meta.get("keywords") or []),
    "license": meta.get("license"),
    "format": meta.get("format"),
}

df = pd.DataFrame(list(flattened.items()), columns=["field", "value"])
df

,field,value
0,@type,Dataset
1,title,2025 Global Climate Data
2,creator,Global Climate Data Team
3,identifier,10.1234/gcd-2025
4,publisher,Open Research Repository (Teaching)
5,description,A small teaching dataset containing fictional ...
6,keywords,climate; temperature; rainfall; CO2 emissions;...
7,license,https://creativecommons.org/licenses/by/4.0/
8,format,text/csv


## Discovery-related fields (teaching highlight)

For Findability, students often focus on:
- `identifier` (persistent DOI-like value)
- `keywords`
- `name` / `title`
- `@type` (here: `Dataset`)

For Reusability, students should also check:
- `description`
- `license`
